# P1-3 · Cây quyết định và ensemble

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q lightgbm

## Lý thuyết

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

# 10 khách hàng của ví dụ trên: cây độ sâu 1 phải chọn đúng ngưỡng 27.5
X10 = pd.DataFrame({"thu_nhap": [8, 12, 15, 20, 25, 30, 35, 40, 50, 60]})
y10 = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1]
stump = DecisionTreeClassifier(max_depth=1).fit(X10, y10)
print(export_text(stump, feature_names=["thu_nhap"]))

# 569 khối u, 30 đặc trưng số, nhãn lành/ác (có sẵn trong scikit-learn, không cần tải)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))   # in cây dạng chữ

|--- thu_nhap <= 27.50
|   |--- class: 0
|--- thu_nhap >  27.50
|   |--- class: 1

|--- worst perimeter <= 106.10
|   |--- worst concave points <= 0.13
|   |   |--- area error <= 48.98
|   |   |   |--- class: 1
|   |   |--- area error >  48.98
|   |   |   |--- class: 0
|   |--- worst concave points >  0.13
|   |   |--- worst texture <= 26.00
|   |   |   |--- class: 1
|   |   |--- worst texture >  26.00
|   |   |   |--- class: 0
|--- worst perimeter >  106.10
|   |--- worst concave points <= 0.14
|   |   |--- worst radius <= 17.61
|   |   |   |--- class: 1
|   |   |--- worst radius >  17.61
|   |   |   |--- class: 0
|   |--- worst concave points >  0.14
|   |   |--- texture error <= 0.43
|   |   |   |--- class: 1
|   |   |--- texture error >  0.43
|   |   |   |--- class: 0



In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=2, n_jobs=-1, oob_score=True, random_state=0)
rf.fit(X_train, y_train)
rf.oob_score_          # ước lượng accuracy không cần tập validation riêng

0.9624413145539906


In [ ]:
import lightgbm as lgb
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=0)
model = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.03, num_leaves=31, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(100)])     # dừng khi 100 cây liền không cải thiện val
model.best_iteration_                              # số cây thật sự dùng

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[122]	valid_0's binary_logloss: 0.134858
122


In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=0)
sorted(zip(r.importances_mean, X_val.columns), reverse=True)[:10]

[(0.03271028037383178, 'worst radius'), (0.012149532710280376, 'worst texture'), (0.008411214953271028, 'worst area'), (0.0009345794392523366, 'worst perimeter'), (0.0, 'worst symmetry'), (0.0, 'worst smoothness'), (0.0, 'worst fractal dimension'), (0.0, 'worst concavity'), (0.0, 'worst compactness'), (0.0, 'texture error')]


## Bài tập 4: Đoán trước rồi kiểm chứng {khái niệm}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
X, y = make_classification(2000, 20, n_informative=6, flip_y=0.1, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(X, y, random_state=0)
for d in [1, 3, 5, 8, 12, 20]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    print(d, round(t.score(Xtr, ytr), 3), round(t.score(Xva, yva), 3))
rf = RandomForestClassifier(300, max_depth=20, n_jobs=-1, random_state=0).fit(Xtr, ytr)
print("RF", rf.score(Xva, yva))

1 0.761 0.754
3 0.836 0.796
5 0.904 0.828
8 0.965 0.81
12 0.996 0.806
20 1.0 0.8
RF 0.868


## Bài tập 5: LightGBM với early stopping {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import lightgbm as lgb, matplotlib.pyplot as plt, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

data = load_breast_cancer(as_frame=True)
Xtr, Xva, ytr, yva = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=0)
m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=15, verbose=-1)
m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
print("số cây tốt nhất:", m.best_iteration_, "| acc val:", m.score(Xva, yva))
pi = permutation_importance(m, Xva, yva, n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=Xtr.columns).nlargest(10).plot.barh(); plt.show()

số cây tốt nhất: 89 | acc val: 0.9300699300699301
